# Kaggle XAI Notebook for ABSA Transformer
Notebook nay chay cac ky thuat giai thich: LIME, SHAP, Integrated Gradients, Attention map, va gradient-based token importance.

In [ ]:
!pip install -q transformers lime shap captum seaborn

In [ ]:
import os
import glob
import numpy as np
import torch
import shap
import seaborn as sns
import matplotlib.pyplot as plt

from lime.lime_text import LimeTextExplainer
from captum.attr import LayerIntegratedGradients
from transformers import AutoTokenizer, AutoModelForSequenceClassification

sns.set_theme(style='whitegrid')

In [ ]:
# Optional: clone repo if needed
REPO_DIR = '/kaggle/working/xai-transformer'
REPO_URL = 'https://github.com/haiyen040602/xai-absa.git'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd /kaggle/working/xai-transformer

In [ ]:
# Model path auto-detection
# Priority 1: model trained in current notebook session
# Priority 2: model attached from /kaggle/input

candidate_paths = []
candidate_paths += glob.glob('/kaggle/working/absa_outputs/transformer')
candidate_paths += glob.glob('/kaggle/input/*/transformer')
candidate_paths += glob.glob('/kaggle/input/*/*transformer*')

MODEL_PATH = None
for p in candidate_paths:
    if os.path.exists(os.path.join(p, 'config.json')):
        MODEL_PATH = p
        break

if MODEL_PATH is None:
    raise FileNotFoundError(
        'Khong tim thay model fine-tuned. Hay train truoc hoac attach model vao Kaggle Input.'
    )

print('Using MODEL_PATH:', MODEL_PATH)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()

name_hint = (model.config._name_or_path or '').lower()
if 'roberta' in name_hint:
    SEP = '</s></s>'
elif 'xlnet' in name_hint:
    SEP = '<sep>'
else:
    SEP = '[SEP]'

label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
print('Separator:', SEP)
print('Device:', device)

In [ ]:
# Example input
text = 'The food was amazing but the service was very slow.'
aspect = 'service'

def build_input(t, a):
    return f'{t} {SEP} {a}'

def predict_proba(texts, aspects):
    all_probs = []
    for t, a in zip(texts, aspects):
        x = build_input(t, a)
        enc = tokenizer(x, return_tensors='pt', truncation=True, padding=True).to(device)
        with torch.no_grad():
            out = model(**enc).logits
            probs = torch.softmax(out, dim=-1).cpu().numpy()[0]
        all_probs.append(probs)
    return np.array(all_probs)

probs = predict_proba([text], [aspect])[0]
pred = int(np.argmax(probs))
print('Input:', build_input(text, aspect))
print('Predicted:', label_map[pred])
print('Probabilities:', probs)

## 1) LIME

In [ ]:
explainer = LimeTextExplainer(class_names=['negative', 'neutral', 'positive'])

def lime_predict(raw_texts):
    aspects = [aspect] * len(raw_texts)
    return predict_proba(raw_texts, aspects)

exp = explainer.explain_instance(text, lime_predict, num_features=10, num_samples=1000, labels=[0, 1, 2])
exp.show_in_notebook(label=exp.top_labels[0])

## 2) SHAP

In [ ]:
def shap_predict(raw_texts):
    inputs = [build_input(t, aspect) for t in raw_texts]
    enc = tokenizer(inputs, return_tensors='pt', truncation=True, padding=True).to(device)
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
    return probs

shap_explainer = shap.Explainer(shap_predict, tokenizer)
shap_values = shap_explainer([text])
shap.initjs()
shap.text_plot(shap_values)

## 3) Integrated Gradients (Captum)

In [ ]:
x = build_input(text, aspect)
enc = tokenizer(x, return_tensors='pt', truncation=True, padding=True).to(device)
input_ids = enc['input_ids']
attention_mask = enc['attention_mask']

with torch.no_grad():
    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    target_class = int(torch.argmax(logits, dim=-1).item())

base_model = model.base_model
lig = LayerIntegratedGradients(
    lambda ids, mask: model(input_ids=ids, attention_mask=mask).logits,
    base_model.embeddings
)

attr, delta = lig.attribute(
    inputs=input_ids,
    baselines=torch.zeros_like(input_ids),
    additional_forward_args=(attention_mask,),
    target=target_class,
    return_convergence_delta=True
)

token_scores = attr.sum(dim=-1).squeeze(0).detach().cpu().numpy()
tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0))

plt.figure(figsize=(12, 4))
sns.barplot(x=list(range(len(tokens))), y=token_scores, color='#4C78A8')
plt.xticks(list(range(len(tokens))), tokens, rotation=90)
plt.title(f'Integrated Gradients - target={label_map[target_class]}')
plt.tight_layout()
plt.show()

print('Convergence delta:', float(delta.mean().item()))

## 4) Attention Weights (last layer averaged over heads)

In [ ]:
att_model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, output_attentions=True).to(device)
att_model.eval()

x = build_input(text, aspect)
enc = tokenizer(x, return_tensors='pt', truncation=True, padding=True).to(device)

with torch.no_grad():
    out = att_model(**enc)

att = out.attentions[-1].mean(dim=1).squeeze(0).detach().cpu().numpy()
tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'].squeeze(0))

plt.figure(figsize=(10, 8))
sns.heatmap(att, cmap='coolwarm')
plt.xticks(np.arange(len(tokens)) + 0.5, tokens, rotation=90)
plt.yticks(np.arange(len(tokens)) + 0.5, tokens, rotation=0)
plt.title('Last-layer Attention (mean over heads)')
plt.tight_layout()
plt.show()

## 5) Gradient-based Token Importance (Grad x Input)

In [ ]:
x = build_input(text, aspect)
enc = tokenizer(x, return_tensors='pt', truncation=True, padding=True).to(device)
input_ids = enc['input_ids']
attention_mask = enc['attention_mask']

emb = model.get_input_embeddings()(input_ids)
emb.retain_grad()

logits = model(inputs_embeds=emb, attention_mask=attention_mask).logits
target = torch.argmax(logits, dim=-1).item()
score = logits[0, target]

model.zero_grad()
score.backward()

grad = emb.grad
token_importance = (grad * emb).sum(dim=-1).squeeze(0).detach().cpu().numpy()
token_importance = np.maximum(token_importance, 0)
if token_importance.max() > 0:
    token_importance = token_importance / token_importance.max()

tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0))

plt.figure(figsize=(12, 4))
sns.barplot(x=list(range(len(tokens))), y=token_importance, color='#F58518')
plt.xticks(list(range(len(tokens))), tokens, rotation=90)
plt.ylim(0, 1.05)
plt.title(f'Gradient-based Token Importance - target={label_map[target]}')
plt.tight_layout()
plt.show()